# DWD ICON precipitation — direct-HTTPS fetch

DWD Open Data publishes ICON forecasts as per-variable, bz2-compressed GRIB2 files over plain HTTPS (no SDK, no auth). earthlens' DWD centre downloads and decompresses them.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
```

!!! note "Native grid"
    DWD's native ICON-global files are on an **icosahedral** grid, which is not a regular lat/lon raster — so this notebook fetches the raw GRIB2 rather than cropping it to a COG. For a croppable COG use a regular-lat/lon ICON product (e.g. `icon-eu`).

## Setup

The imports for the notebook. `earthlens.nwp` provides the `Catalog` of forecast models and the `DWDCentre` that fetches DWD Open Data GRIB2 files directly over HTTPS.

In [ ]:
import datetime as dt
from pathlib import Path

from earthlens.nwp.centres.dwd import DWDCentre

from earthlens.nwp import Catalog

## 1 · The ICON-global model

Look up the `icon-global` model in the catalog and inspect its band table. Each entry maps an earthlens variable name to the DWD GRIB short name (e.g. `precipitation_acc` → `TOT_PREC`).

In [ ]:
model = Catalog().get_model('icon-global')
model.bands

## 2 · Pick a published forecast cycle

ICON-global runs at 00/06/12/18Z and DWD keeps only roughly the last day online, so we pick the most recent run that is already published: back off about 7 hours and floor to the 6-hourly cycle grid.

In [ ]:
# ICON-global runs 00/06/12/18Z and DWD keeps only ~the last day online,
# so pick the most recent run that is already published (~back off 7 h,
# floored to the 6-hourly cycle grid).
now = dt.datetime.now(dt.UTC).replace(tzinfo=None)
ref = now - dt.timedelta(hours=7)
cycle = ref.replace(hour=(ref.hour // 6) * 6, minute=0, second=0, microsecond=0)
cycle

## 3 · Fetch the precipitation GRIB2

Create the output directory, then ask the `DWDCentre` to fetch a single GRIB2 file: the `precipitation_acc` band of analysis step `0` for the chosen cycle. `mirror='auto'` selects a working DWD mirror; the call returns the path to the decompressed `.grib2`.

In [ ]:
out_dir = Path('out/icon')
out_dir.mkdir(parents=True, exist_ok=True)

Run the fetch and inspect the resulting file's name and on-disk size.

In [ ]:
centre = DWDCentre(out_dir)
grib_path = centre.fetch_one(
    model, cycle, step=0, params=['precipitation_acc'], mirror='auto'
)
grib_path.name, grib_path.stat().st_size

## Recap

The downloaded `.grib2` holds the requested band's decompressed messages. For a regular-grid model you would instead call `EarthLens(data_source="nwp", variables={...}).download()` and receive a bbox-cropped COG, exactly like the GFS quickstart.